Importing the dataset

In [10]:
import pandas as pd

food = pd.read_csv("food.csv")
food_nutrient = pd.read_csv("food_nutrient.csv")
nutrient = pd.read_csv("nutrient.csv")

food.head()


C:\Users\Vandan Agrawal\AppData\Local\Temp\ipykernel_29124\763829690.py:4: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  food_nutrient = pd.read_csv("food_nutrient.csv")


,fdc_id,data_type,description,food_category_id,publication_date
0,319874,sample_food,"HUMMUS, SABRA CLASSIC",16.0,2019-04-01
1,319875,market_acquisition,"HUMMUS, SABRA CLASSIC",16.0,2019-04-01
2,319876,market_acquisition,"HUMMUS, SABRA CLASSIC",16.0,2019-04-01
3,319877,sub_sample_food,Hummus,16.0,2019-04-01
4,319878,sub_sample_food,Hummus,16.0,2019-04-01


Just keeping raw ingredients and removing store bought packaged foods for more accuracy

In [11]:
food_filtered = food[
    food["description"].str.contains(
        "raw|fresh|oil|flour|cheese|milk|egg|tomato|onion|garlic",
        case=False,
        na=False
    )
]


In [12]:
food_filtered["description"].sample(20)


75944                        Peppers, serrano, seeded, raw
21582        ALMOND MILK, UNSWEETENED, PLAIN, SHELF STABLE
43075                              blueberries, fresh, raw
42757                    grapes, red, seedless, fresh, raw
53337                                 Chicken, ground, raw
19590                                        ONIONS, WHITE
74068             Squash, pie pumpkin, peeled, seeded, raw
12503                                     Tomatoes, canned
29306           SOY MILK, UNSWEETENED, PLAIN, SHELF STABLE
66099                 Kiwifruit (kiwi), green, peeled, raw
8075     Pantothenic Acid, Cottage cheese, 2% milkfat, ...
42969                  grapes, green, seedless, fresh, raw
47449                    Potatoes, gold, without skin, raw
19446                                   FLOUR, RICE, BROWN
62960               tomato, sauce, canned, with salt added
71590                          Beef, tenderloin steak, raw
32001                            flour, spelt, whole gra

Building ingredient table list 

In [13]:
# STEP B: Build ingredient lookup table from USDA FDC Foundation Foods
# Produces: ingredient_lookup.csv with fdc_id -> canonical ingredient
# Also includes the original FDC description for sanity checks.

import re
import pandas as pd

# -----------------------------
# 1) Load FDC files (Foundation Foods)
# -----------------------------
FOOD_CSV_PATH = "food.csv"  # adjust path if needed
food = pd.read_csv(FOOD_CSV_PATH)

# Safety: keep only columns we need (avoid huge memory use)
needed_cols = [c for c in ["fdc_id", "description", "data_type", "food_category_id"] if c in food.columns]
food = food[needed_cols].copy()

# -----------------------------
# 2) Quick filter to reduce obviously non-ingredient entries
#    (Foundation Foods is already ingredient-focused, but still has some prepared items)
# -----------------------------
# You can relax/tighten this later. For now: keep rows with a description.
food = food.dropna(subset=["description"]).copy()
food["description"] = food["description"].astype(str)

# Optional: if data_type exists, keep Foundation only
if "data_type" in food.columns:
    # Foundation foods dataset typically uses "foundation_food"
    food = food[food["data_type"].str.contains("foundation", case=False, na=False)].copy()

# -----------------------------
# 3) Canonical ingredient mapping (expandable)
#    Order matters: more specific patterns should go first.
# -----------------------------
PATTERN_TO_CANONICAL = [
    # Oils & fats
    (r"\bolive oil\b|\boil, olive\b", "olive_oil"),
    (r"\bcanola oil\b", "canola_oil"),
    (r"\bvegetable oil\b", "vegetable_oil"),
    (r"\bbutter\b", "butter"),

    # Core Italian vegetables
    (r"\btomato(es)?\b", "tomato"),
    (r"\bonion(s)?\b", "onion"),
    (r"\bgarlic\b", "garlic"),
    (r"\bbasil\b", "basil"),
    (r"\boregano\b", "oregano"),
    (r"\bparsley\b", "parsley"),
    (r"\brosemary\b", "rosemary"),
    (r"\bthyme\b", "thyme"),
    (r"\bspinach\b", "spinach"),
    (r"\bmushroom(s)?\b", "mushroom"),
    (r"\bzucchini\b", "zucchini"),
    (r"\beggplant\b|\baubergine\b", "eggplant"),
    (r"\bpepper\b", "pepper"),
    (r"\bbroccoli\b", "broccoli"),
    (r"\bkale\b", "kale"),

    # Fruits commonly used
    (r"\blemon\b", "lemon"),
    (r"\bbanana(s)?\b", "banana"),
    (r"\bblueberr(y|ies)\b", "blueberry"),
    (r"\bpeach(es)?\b", "peach"),
    (r"\bavocado\b", "avocado"),
    (r"\bapricot(s)?\b", "apricot"),
    (r"\bkiwi(fruit)?\b", "kiwi"),
    (r"\bcantaloupe\b|\bmelon\b", "melon"),

    # Beans & legumes
    (r"\bbeans?\b", "beans"),
    (r"\bkidney beans?\b", "kidney_beans"),
    (r"\bblack beans?\b", "black_beans"),
    (r"\bpinto beans?\b", "pinto_beans"),
    (r"\blentils?\b", "lentils"),
    (r"\bchickpeas?\b|\bgarbanzo\b", "chickpeas"),

    # Grains & pasta
    (r"\bpasta\b|\bspaghetti\b|\bpenne\b|\bfusilli\b|\bmacaroni\b", "pasta"),
    (r"\brace\b", "rice"),
    (r"\bflour\b", "flour"),

    # Dairy
    (r"\bparmesan\b|\bparmigiano\b", "parmesan_cheese"),
    (r"\bmozzarella\b", "mozzarella"),
    (r"\bricotta\b", "ricotta"),
    (r"\bpecorino\b", "pecorino"),
    (r"\bmilk\b", "milk"),
    (r"\byogurt\b", "yogurt"),

    # Protein sources
    (r"\begg(s)?\b", "egg"),
    (r"\bchicken\b", "chicken"),
    (r"\bbeef\b", "beef"),
    (r"\bpork\b", "pork"),
    (r"\bsalmon\b", "salmon"),
    (r"\btuna\b", "tuna"),
    (r"\bsnapper\b", "snapper"),
    (r"\bswordfish\b", "swordfish"),
    (r"\bbison\b", "bison"),

    # Nuts
    (r"\bpecans?\b", "pecans"),
    (r"\bpine nuts?\b", "pine_nuts"),
    (r"\bwalnuts?\b", "walnuts"),
    (r"\balmonds?\b", "almonds"),

    # Pantry basics
    (r"\bsalt\b", "salt"),
    (r"\bsugar(s)?\b", "sugar"),
]


# Precompile regexes for speed
COMPILED = [(re.compile(pat, flags=re.IGNORECASE), canon) for pat, canon in PATTERN_TO_CANONICAL]

def normalize_ingredient(desc: str) -> str | None:
    """Map an FDC description string to a canonical ingredient name."""
    if not isinstance(desc, str) or not desc.strip():
        return None

    d = desc.lower()

    for rx, canon in COMPILED:
        if rx.search(d):
            return canon

    return None  # unknown/unmapped

# -----------------------------
# 4) Apply mapping
# -----------------------------
food["ingredient"] = food["description"].apply(normalize_ingredient)

# Keep both mapped and unmapped so you can inspect what you're missing
mapped = food.dropna(subset=["ingredient"]).copy()
unmapped = food[food["ingredient"].isna()].copy()

# -----------------------------
# 5) Write outputs
# -----------------------------
# Main lookup (mapped only)
ingredient_lookup = mapped[["fdc_id", "ingredient", "description"]].drop_duplicates()
ingredient_lookup.to_csv("ingredient_lookup.csv", index=False)

# Helpful debug file: what didn't map
unmapped[["fdc_id", "description"]].drop_duplicates().to_csv("ingredient_unmapped_debug.csv", index=False)

# Summary print
print("✅ ingredient_lookup.csv saved")
print("Rows (mapped):", ingredient_lookup.shape[0])
print("Unique ingredients:", ingredient_lookup["ingredient"].nunique())
print("\nTop canonical ingredients:")
print(ingredient_lookup["ingredient"].value_counts().head(20))

print("\n✅ ingredient_unmapped_debug.csv saved")
print("Rows (unmapped):", unmapped.shape[0])


✅ ingredient_lookup.csv saved
Rows (mapped): 243
Unique ingredients: 42

Top canonical ingredients:
ingredient
beans        47
flour        30
beef         24
milk         18
mushroom     15
pork         13
tomato       10
chicken       9
egg           9
butter        6
banana        5
onion         5
spinach       4
sugar         3
salt          3
olive_oil     3
yogurt        3
broccoli      2
kale          2
ricotta       2
Name: count, dtype: int64

✅ ingredient_unmapped_debug.csv saved
Rows (unmapped): 193


Attaching nutrient values to the ingredients

In [14]:
import pandas as pd

# -----------------------------
# Load files
# -----------------------------
ingredient_lookup = pd.read_csv("ingredient_lookup.csv")
food_nutrient = pd.read_csv("food_nutrient.csv")
nutrient = pd.read_csv("nutrient.csv")

# -----------------------------
# Keep nutrients we need
# -----------------------------
wanted_nutrients = {
    "Energy": "calories",
    "Protein": "protein",
    "Total lipid (fat)": "fat",
    "Carbohydrate, by difference": "carbs"
}

nutrient_subset = nutrient[
    nutrient["name"].isin(wanted_nutrients.keys())
][["id", "name"]]

nutrient_subset["nutrient_name"] = nutrient_subset["name"].map(
    wanted_nutrients
)

# -----------------------------
# Merge nutrient values
# -----------------------------
merged = food_nutrient.merge(
    nutrient_subset,
    left_on="nutrient_id",
    right_on="id"
)

merged = merged.merge(
    ingredient_lookup,
    on="fdc_id"
)

# -----------------------------
# Keep needed columns
# -----------------------------
ingredient_nutrients = merged[
    ["ingredient", "nutrient_name", "amount"]
]

# -----------------------------
# Aggregate per ingredient
# -----------------------------
ingredient_nutrients = (
    ingredient_nutrients
    .groupby(["ingredient", "nutrient_name"])
    ["amount"]
    .mean()
    .reset_index()
)

# -----------------------------
# Save table
# -----------------------------
ingredient_nutrients.to_csv(
    "ingredient_nutrient_table.csv",
    index=False
)

print("✅ Ingredient nutrient table created")
print(ingredient_nutrients.head())


✅ Ingredient nutrient table created
  ingredient nutrient_name      amount
0    almonds      calories  1605.00000
1    almonds         carbs    18.11731
2    almonds           fat    54.44500
3    almonds       protein    20.92519
4    apricot         carbs    10.23875


C:\Users\Vandan Agrawal\AppData\Local\Temp\ipykernel_29124\2235284763.py:7: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  food_nutrient = pd.read_csv("food_nutrient.csv")


Actually creating the FOPCs

In [15]:
import pandas as pd
from pathlib import Path

INPUT_CSV = "ingredient_nutrient_table.csv"

OUT_DIR = Path(".")
ING_FACTS_PATH = OUT_DIR / "facts_ingredients.pl"
NUTR_FACTS_PATH = OUT_DIR / "facts_nutrients.pl"

def safe_atom(s: str) -> str:
    s = str(s).strip().lower()
    cleaned = []
    for ch in s:
        if ch.isalnum():
            cleaned.append(ch)
        else:
            cleaned.append("_")
    s = "".join(cleaned)

    while "__" in s:
        s = s.replace("__", "_")

    s = s.strip("_")
    if not s:
        return "unknown"
    if s[0].isdigit():
        s = "n_" + s
    return s

def format_float(x) -> str:
    try:
        fx = float(x)
    except Exception:
        return "0"
    return f"{fx:.6g}"

df = pd.read_csv(INPUT_CSV)

df["ingredient_atom"] = df["ingredient"].apply(safe_atom)
df["nutrient_atom"] = df["nutrient_name"].apply(safe_atom)

unique_ings = sorted(df["ingredient_atom"].unique())

with open(ING_FACTS_PATH, "w", encoding="utf-8") as f:
    f.write("% Ingredient facts\n")
    for ing in unique_ings:
        f.write(f"Ingredient({ing}).\n")

with open(NUTR_FACTS_PATH, "w", encoding="utf-8") as f:
    f.write("% Nutrient facts per 100g\n")
    for _, row in df.iterrows():
        ing = row["ingredient_atom"]
        nutr = row["nutrient_atom"]
        amt = format_float(row["amount"])
        f.write(f"HasNutrient({ing}, {nutr}).\n")
        f.write(f"NutrientValuePer100g({ing}, {nutr}, {amt}).\n")

print("Facts generated successfully!")


Facts generated successfully!


## Loading all ingredients (Data set and recipe ingredents)

**Allergen tagging (Top 9)**  
Run the cell below to build `facts_allergens.pl` from `all_ingredients_current.csv`. Only ingredients that contain any Top 9 allergen (milk, egg, fish, shellfish, tree nuts, peanut, wheat, soy, sesame) are tagged; the rest stay untagged. Load `facts_allergens.pl` in the next cell to query `HasAllergen(ingredient, allergen)` later.

In [36]:
# Generate facts_allergens.pl from all_ingredients_current.csv (Top 9 allergens only)
# Run this when the ingredient list changes. Then load facts_allergens.pl in the next cell.

import pandas as pd
import re

TOP9_ALLERGENS = {
    "milk": ["milk", "butter", "creme_fraiche", "fromage_frais", "mascarpone", "mozzarella",
             "parmesan_cheese", "parmigiano_reggiano", "pecorino", "ricotta", "yogurt"],
    "egg": ["egg", "egg_yolks"],
    "fish": ["anchovy", "salmon", "pilchards", "snapper", "swordfish", "tuna"],
    "shellfish": ["prawn", "shrimp", "crab", "lobster", "crayfish"],
    "tree_nuts": ["almonds", "pecans", "pine_nuts", "walnuts", "cashew", "hazelnut", "macadamia"],
    "peanut": ["peanut"],
    "wheat": ["flour", "bread", "pasta", "spaghetti", "penne", "rigatoni", "farfalle",
              "lasagna", "linguine", "paccheri", "bowtie"],
    "soy": ["soy_milk", "soy"],
    "sesame": ["sesame", "tahini"],
}

def normalize_ingredient_from_csv(s):
    """Extract ingredient atom from CSV value like \"('almonds',)\"."""
    s = str(s).strip().strip('"')
    m = re.search(r"['\"]([a-z0-9_]+)['\"]", s)
    return m.group(1) if m else s.replace("(", "").replace(")", "").replace("'", "").replace(",", "").strip()

def get_allergens_for_ingredient(ing: str):
    """Return list of (ingredient, allergen) for Top 9 only; empty if none."""
    ing_lower = ing.lower()
    out = []
    for allergen, keywords in TOP9_ALLERGENS.items():
        if any(kw in ing_lower for kw in keywords):
            out.append((ing, allergen))
    return out

# Load current ingredients
df = pd.read_csv("all_ingredients_current.csv")
ingredients = df["ingredient"].apply(normalize_ingredient_from_csv).dropna().unique().tolist()

# Build HasAllergen(ingredient, allergen). only for Top 9
lines = [
    "% ALLERGEN FACTS (Top 9). Only tagged ingredients; rest untagged.",
    "% HasAllergen(ingredient, allergen).",
    "%"
]
tagged = set()
for ing in sorted(ingredients):
    for ing_atom, allergen in get_allergens_for_ingredient(ing):
        lines.append(f"HasAllergen({ing_atom}, {allergen}).")
        tagged.add(ing_atom)

with open("facts_allergens.pl", "w", encoding="utf-8") as f:
    f.write("\n".join(lines) + "\n")

print(f"facts_allergens.pl written. Tagged {len(tagged)} ingredients with Top 9 allergens.")
print("Untagged (no Top 9):", len(ingredients) - len(tagged), "ingredients.")

facts_allergens.pl written. Tagged 56 ingredients with Top 9 allergens.
Untagged (no Top 9): 126 ingredients.


In [37]:
from pyDatalog import pyDatalog
import re

pyDatalog.clear()

# Declare terms you’ll use
pyDatalog.create_terms("""
    Ingredient, HasNutrient, NutrientValuePer100g,
    IngredientAlias, CanonicalIngredient, IgnoredIngredient,
    HasNutrientResolved, ResolvedNutrientValuePer100g,
    HasAllergen,
    I, C, N, V
""")

def load_prolog_style_facts(file_path: str):
    """
    Loads facts written like:
      Predicate(arg1, arg2, ...).
    where args are atoms (strings without quotes) or numbers.
    Supports any predicate name.
    """
    fact_re = re.compile(r"([A-Za-z_]\w*)\(([^)]+)\)\.?")

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("%"):
                continue
            for m in fact_re.finditer(line):
                pred = m.group(1)
                args_str = m.group(2).strip()

            # Split by commas (your facts don’t contain quoted strings, so safe)
                raw_args = [a.strip() for a in args_str.split(",") if a.strip()]
                args = []
                for a in raw_args:
                    try:
                        args.append(float(a))
                    except ValueError:
                        args.append(a)

                
                if pred not in globals():
                    continue

                predicate_obj = globals()[pred]
                +predicate_obj(*args)

# Load your files
load_prolog_style_facts("facts_ingredients.pl")
load_prolog_style_facts("facts_nutrients.pl")
load_prolog_style_facts("facts_standardized_overrides.pl")
load_prolog_style_facts("facts_allergens.pl")

# Alias-resolution rules
ResolvedNutrientValuePer100g(I, N, V) <= NutrientValuePer100g(I, N, V)
ResolvedNutrientValuePer100g(I, N, V) <= IngredientAlias(I, C) & NutrientValuePer100g(C, N, V)

HasNutrientResolved(I, N) <= HasNutrient(I, N)
HasNutrientResolved(I, N) <= IngredientAlias(I, C) & HasNutrient(C, N)

print("✅ Loaded facts (Prolog-style) + enabled alias resolution + allergens.")

✅ Loaded facts (Prolog-style) + enabled alias resolution + allergens.


In [ ]:
**Example output:** See the **final block at the end of the notebook** for a printed example of several ingredients with their tags (alias/base), nutritional values per 100g, and allergens.

In [38]:
import pandas as pd
import re

# ---- Paths (edit if needed) ----
ING_LOOKUP_PATH = "ingredient_lookup.csv"
OUT_SELECTED_PATH = "ingredient_lookup_selected.csv"

df = pd.read_csv(ING_LOOKUP_PATH)

NEGATIVE_TERMS = [
    "canned", "cooked", "fried", "roasted", "grilled", "baked",
    "frozen", "prepared", "with", "added", "sweetened", "in sauce",
    "pickled", "smoked", "breaded", "seasoned", "dried", "dehydrated"
]

POSITIVE_TERMS = [
    "raw", "fresh"
]

# Optional ingredient-specific boosts (helps pick a more “standard” entry)
PREFERRED_REGEX = {
    "tomato": [r"tomatoes, red, ripe, raw", r"tomato.*raw"],
    "onion":  [r"onions?, raw"],
    "garlic": [r"garlic.*raw"],
    "olive_oil": [r"oil, olive(,|$)"],
}

def score_description(ingredient: str, desc: str) -> float:
    d = str(desc).lower()
    score = 0.0

    # Positive boosts
    for t in POSITIVE_TERMS:
        if t in d:
            score += 5.0 if t == "raw" else 2.0

    # Negative penalties
    for t in NEGATIVE_TERMS:
        if t in d:
            score -= 4.0

    # Ingredient-specific preferences
    if ingredient in PREFERRED_REGEX:
        for rx in PREFERRED_REGEX[ingredient]:
            if re.search(rx, d):
                score += 10.0

    # Tie-breaker: shorter descriptions usually more “basic”
    score += max(0, 3.0 - (len(d) / 80.0))  # small bonus for shorter

    return score

df["score"] = df.apply(lambda r: score_description(r["ingredient"], r["description"]), axis=1)

# Pick best row per canonical ingredient
selected = (
    df.sort_values(["ingredient", "score"], ascending=[True, False])
      .groupby("ingredient", as_index=False)
      .head(1)
      .reset_index(drop=True)
)

selected.to_csv(OUT_SELECTED_PATH, index=False)

print("✅ Saved:", OUT_SELECTED_PATH)
print("Unique ingredients:", selected["ingredient"].nunique())
print("\nSample selections:")
print(selected[["ingredient", "fdc_id", "description", "score"]].head(15))


✅ Saved: ingredient_lookup_selected.csv
Unique ingredients: 42

Sample selections:
   ingredient   fdc_id                                        description  \
0     almonds  2346393                          Nuts, almonds, whole, raw   
1     apricot  2710815                            Apricot, with skin, raw   
2     avocado  2710824                         Avocado, Hass, peeled, raw   
3      banana   790774                             Bananas, overripe, raw   
4       beans  2346400                            Beans, snap, green, raw   
5        beef  2727573                        Beef, tenderloin steak, raw   
6       bison  2727571                                 Bison, ground, raw   
7   blueberry  2263889                                   Blueberries, raw   
8    broccoli   321900                                      Broccoli, raw   
9      butter   790508                              Butter, stick, salted   
10    chicken  2727568                  Chicken, wing, meat and skin, 

In [39]:
def print_fopc_examples(file_path, n=15):
    print(f"\n--- Sample facts from {file_path} ---\n")
    
    with open(file_path, "r", encoding="utf-8") as f:
        count = 0
        for line in f:
            if line.startswith("%"):
                    continue
            print(line.strip())
            count += 1
            if count >= n:
                break

print_fopc_examples("facts_ingredients.pl", 20)
print_fopc_examples("facts_nutrients.pl", 20)



--- Sample facts from facts_ingredients.pl ---

Ingredient(almonds).
Ingredient(apricot).
Ingredient(avocado).
Ingredient(banana).
Ingredient(beans).
Ingredient(beef).
Ingredient(bison).
Ingredient(blueberry).
Ingredient(broccoli).
Ingredient(butter).
Ingredient(chicken).
Ingredient(egg).
Ingredient(eggplant).
Ingredient(flour).
Ingredient(garlic).
Ingredient(kale).
Ingredient(kiwi).
Ingredient(lentils).
Ingredient(melon).
Ingredient(milk).

--- Sample facts from facts_nutrients.pl ---

HasNutrient(almonds, calories).
NutrientValuePer100g(almonds, calories, 1605).
HasNutrient(almonds, carbs).
NutrientValuePer100g(almonds, carbs, 18.1173).
HasNutrient(almonds, fat).
NutrientValuePer100g(almonds, fat, 54.445).
HasNutrient(almonds, protein).
NutrientValuePer100g(almonds, protein, 20.9252).
HasNutrient(apricot, carbs).
NutrientValuePer100g(apricot, carbs, 10.2387).
HasNutrient(apricot, fat).
NutrientValuePer100g(apricot, fat, 0.405).
HasNutrient(apricot, protein).
NutrientValuePer100g(apr

## Adding the alergens

In [41]:
from pyDatalog import pyDatalog

# Declare allergen predicates
pyDatalog.create_terms("""
    ContainsAllergen, ContainsAllergenResolved,
    Allergen,
    A
""")

# List top 9 allergens (as atoms)
TOP9 = [
    "milk", "egg", "fish", "crustacean_shellfish",
    "tree_nuts", "peanuts", "wheat", "soybeans", "sesame"
]

# Add simple facts that these are allergens (optional, but nice)
for a in TOP9:
    +Allergen(a)

# IMPORTANT: Resolve allergens through aliases (same idea as nutrients)
# If an ingredient is an alias of a canonical ingredient, it inherits allergens too.
ContainsAllergenResolved(I, A) <= ContainsAllergen(I, A)
ContainsAllergenResolved(I, A) <= IngredientAlias(I, C) & ContainsAllergen(C, A)

print("✅ Allergen predicates ready (top 9) + alias resolution enabled.")

✅ Allergen predicates ready (top 9) + alias resolution enabled.


In [42]:
# Helper: add multiple allergen facts quickly
def add_allergens(ingredient, allergens):
    for a in allergens:
        +ContainsAllergen(ingredient, a)

# -----------------------
# DAIRY (milk)
# -----------------------
dairy_items = [
    "parmigiano_reggiano", "pecorino", "pecorino_cheese",
    "mascarpone", "mozzarella_balls", "creme_fraiche", "fromage_frais",
    #"vegan_butter"  # NOTE: vegan butter *usually* has no milk, but some do; assume NO milk unless you want strict warnings.
]

# If you want vegan_butter to be treated as dairy-free, comment it out above.
for ing in dairy_items:
    if ing != "vegan_butter":  # assume dairy-free for vegan butter
        add_allergens(ing, ["milk"])

# soy milk
add_allergens("soy_milk", ["soybeans"])

# -----------------------
# EGGS
# -----------------------
add_allergens("egg_yolks", ["egg"])

# -----------------------
# WHEAT (gluten proxy via wheat)
# -----------------------
wheat_items = [
    "spaghetti", "penne", "rigatoni", "paccheri_pasta", "linguine_pasta",
    "farfalle", "bowtie_pasta", "lasagna_noodles",
    "bread", "wholegrain_bread"
]
for ing in wheat_items:
    add_allergens(ing, ["wheat"])

# -----------------------
# FISH
# -----------------------
fish_items = ["anchovy_fillet", "pilchards"]
for ing in fish_items:
    add_allergens(ing, ["fish"])

# -----------------------
# CRUSTACEAN SHELLFISH
# -----------------------
add_allergens("king_prawns", ["crustacean_shellfish"])

# -----------------------
# Condiments & others (usually none of top9 by default)
# Worcestershire sauce can contain anchovy in some brands → fish
# If you want strict safety: tag as fish.
add_allergens("worcestershire_sauce", ["fish"])

print("✅ Added allergen facts for known items.")

✅ Added allergen facts for known items.


In [43]:
# Gather all known ingredients
all_ing_query = Ingredient(I)
all_ingredients = sorted(list(all_ing_query.data))

print("Total Ingredient(...) facts:", len(all_ingredients))
print("\nFirst 60 ingredients:")
print(all_ingredients[:60])

Total Ingredient(...) facts: 182

First 60 ingredients:
[('almonds',), ('anchovy_fillet',), ('anchovy_fillet_std',), ('apricot',), ('asparagus',), ('asparagus_std',), ('avocado',), ('baby_plum_tomatoes',), ('bacon',), ('bacon_std',), ('banana',), ('basil',), ('bay_leaves',), ('beans',), ('beef',), ('beef_stock',), ('beef_stock_std',), ('bison',), ('black_olives',), ('black_pepper',), ('blueberry',), ('bowtie_pasta',), ('bread',), ('bread_white_std',), ('broccoli',), ('butter',), ('cannellini_beans',), ('cannellini_beans_std',), ('carrots',), ('carrots_std',), ('caster_sugar',), ('celery',), ('celery_std',), ('cherry_tomatoes',), ('chicken',), ('chicken_breasts',), ('chicken_breasts_std',), ('chicken_stock',), ('chicken_stock_cube',), ('chicken_stock_cube_std',), ('chicken_stock_std',), ('chopped_tomatoes',), ('citrus_std',), ('creme_fraiche',), ('creme_fraiche_std',), ('dark_rum',), ('dark_rum_std',), ('dry_pasta',), ('dry_white_wine',), ('egg',), ('egg_yolks',), ('egg_yolks_std',), ('

In [44]:
#saving all ingredients
import pandas as pd
pd.DataFrame({"ingredient": all_ingredients}).to_csv("all_ingredients_current.csv", index=False)
print("✅ Saved all_ingredients_current.csv")

✅ Saved all_ingredients_current.csv


In [46]:
# =============================================================================
# EXAMPLE BLOCK: Sample ingredients with tags, nutrition (per 100g), and allergens
# Run after loading all fact files. Output appears in this block.
# =============================================================================

from pyDatalog import pyDatalog

def first_values(answers):
    if not answers or answers in (True, None):
        return []
    return [t[0] if isinstance(t, (list, tuple)) and len(t) >= 1 else t for t in (answers if isinstance(answers, list) else [answers])]

EXAMPLES = ["almonds", "spaghetti", "milk", "salmon", "tomato", "egg", "flour"]

print("=" * 70)
print("EXAMPLE: Ingredients with tags, nutrition (per 100g), and allergens")
print("=" * 70)

for ing in EXAMPLES:
    print(f"\n--- {ing} ---")
    try:
        r = pyDatalog.ask(f"IngredientAlias('{ing}', C)")
        vals = first_values(r.answers if r else None)
        if vals:
            print(f"  Tag: alias -> canonical = {vals}")
        else:
            print(f"  Tag: base ingredient")
    except Exception:
        print(f"  Tag: base ingredient")

    print("  Nutrition (per 100g):")
    for nut in ["calories", "protein", "fat", "carbs"]:
        try:
            r = pyDatalog.ask(f"ResolvedNutrientValuePer100g('{ing}', '{nut}', V)")
            vals = first_values(r.answers if r else None)
            if vals:
                print(f"    {nut}: {vals[0]}")
        except Exception:
            pass

    try:
        r = pyDatalog.ask(f"HasAllergen('{ing}', A)")
        vals = first_values(r.answers if r else None)
        print(f"  Allergens: {vals if vals else '(none / untagged)'}")
    except Exception:
        print(f"  Allergens: (none / untagged)")

print("\n" + "=" * 70)

EXAMPLE: Ingredients with tags, nutrition (per 100g), and allergens

--- almonds ---
  Tag: base ingredient
  Nutrition (per 100g):
    calories: 1605.0
    protein: 20.9252
    fat: 54.445
    carbs: 18.1173
  Allergens: ['tree_nuts']

--- spaghetti ---
  Tag: alias -> canonical = ['dry_pasta']
  Nutrition (per 100g):
    calories: 371.0
    protein: 13.0
    fat: 1.5
    carbs: 75.0
  Allergens: ['wheat']

--- milk ---
  Tag: base ingredient
  Nutrition (per 100g):
    calories: 121.25
    protein: 3.97911
    fat: 2.95411
    carbs: 3.73499
  Allergens: ['milk']

--- salmon ---
  Tag: base ingredient
  Nutrition (per 100g):
    protein: 21.3094
    fat: 9.023
    carbs: 0.0
  Allergens: ['fish']

--- tomato ---
  Tag: base ingredient
  Nutrition (per 100g):
    calories: 58.5
    protein: 1.339
    fat: 0.42513
    carbs: 6.68066
  Allergens: (none / untagged)

--- egg ---
  Tag: base ingredient
  Nutrition (per 100g):
    calories: 759.556
    protein: 26.6111
    fat: 18.9189
    

In [45]:
def show_ingredient_allergens(ingredient):
    res = ContainsAllergenResolved(ingredient, A)
    allergens = sorted(list(res.data)) if hasattr(res, "data") else []
    return allergens

demo_examples = [
    "spaghetti",
    "pecorino_cheese",
    "soy_milk",
    "egg_yolks",
    "king_prawns",
    "worcestershire_sauce",
    "tomatoes",
    "olive_oil"  # likely no allergens in our map
]

for ing in demo_examples:
    print(f"{ing:22s} -> {show_ingredient_allergens(ing)}")

spaghetti              -> [('wheat',)]
pecorino_cheese        -> [('milk',)]
soy_milk               -> [('soybeans',)]
egg_yolks              -> [('egg',)]
king_prawns            -> [('crustacean_shellfish',)]
worcestershire_sauce   -> [('fish',)]
tomatoes               -> []
olive_oil              -> []
